# 01 Data Understanding

This notebook orients the project around the raw Dubai Arab Bank case-study dataset and the application-time credit risk use case.


## Reproducibility Note

This notebook is a lightweight narrative companion. The canonical reproducible workflow lives in `src/`, and generated metrics, figures, leakage audit outputs, explainability artifacts, and fairness reports live in `reports/`. The final recommended model is `xgboost_application.pkl`; behavioral and full-diagnostic models are retained only for monitoring or leakage-diagnostic discussion.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
DATA_PATH = next((ROOT / 'data' / 'raw').glob('*'))
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')

df = pd.read_excel(DATA_PATH)
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
display(df.head())
display(df.dtypes.rename('dtype').to_frame())


## Business framing

The raw data mixes borrower profile, loan terms, macroeconomic context, and post-loan behavioral signals. That mix is useful for auditing, but only application-time variables are valid for the final underwriting model.


In [ ]:
summary = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'unique_values': df.nunique(dropna=False),
}).sort_values(by='missing_values', ascending=False)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Default_Flag'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#7fb069', '#d95d39'])
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('Default_Flag')

df['LoanType'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='#457b9d')
axes[1].set_title('Loan Type Mix')
axes[1].set_xlabel('Loan Type')
plt.tight_layout()


## Takeaways

- The dataset contains **10,000 rows** and **28 original columns**.
- `Default_Flag` is the binary target.
- `CustomerID` and `LoanID` are record identifiers, not model features.
- Behavioral fields such as repayment history belong to a monitoring model, not the final application-time score.
